# Proyecto 1 - Diagnóstico inicial de los datos crudos
Establecimientos educativos hasta el nivel DIVERSIFICADO (MINEDUC, todo el país)

Este notebook:
1. Carga los 22 archivos crudos (uno por departamento / Ciudad Capital) desde `data/raw/`.
2. Los une en un solo DataFrame (sin modificar los archivos originales).
3. Genera el diagnóstico inicial

In [55]:
import pandas as pd
import numpy as np
import glob, os, re
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RAW_DIR = "../data/raw"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Archivos crudos encontrados: {len(files)}")
for f in files:
    print(" -", os.path.basename(f))

Archivos crudos encontrados: 23
 - ALTAVERAPAZ.csv
 - BAJAVERAPAZ.csv
 - CHIMALTENANGO.csv
 - CHIQUIMULA.csv
 - CIUDADCAPITAL.csv
 - ELPROGRESO.csv
 - ESCUINTLA.csv
 - GUATEMALA.csv
 - HUEHUETENANGO.csv
 - IZABAL.csv
 - JALAPA.csv
 - JUTIAPA.csv
 - PETEN.csv
 - QUETZALTENANGO.csv
 - QUICHE.csv
 - RETALHULEU.csv
 - SACATEPEQUEZ.csv
 - SANMARCOS.csv
 - SANTAROSA.csv
 - SOLOLA.csv
 - SUCHITEPEQUEZ.csv
 - TOTONICAPAN.csv
 - ZACAPA.csv


## 1. Carga y unión de los datos crudos

In [56]:
dfs = []
for f in files:
    df = pd.read_csv(f, dtype=str, encoding="utf-8-sig")
    df["__archivo_origen"] = os.path.basename(f)
    dfs.append(df)

datos = pd.concat(dfs, ignore_index=True)
print("Filas totales (incluye posibles filas vacías por archivo):", datos.shape[0])
print("Columnas:", datos.shape[1])

Filas totales (incluye posibles filas vacías por archivo): 11890
Columnas: 18


### 1.1 Filas completamente vacías (residuo de exportación de cada CSV)
Cada archivo exportado desde el portal de MINEDUC trae una fila final vacía.
Se identifico pero no se eliminan en esta etapa (el diagnóstico se realizó sobre los datos tal cual llegaron).

In [57]:
filas_vacias = datos.drop(columns="__archivo_origen").isna().all(axis=1).sum()
tabla_vacias = pd.DataFrame({
    "métrica": ["Filas completamente vacías", "Filas con contenido"],
    "valor": [filas_vacias, datos.shape[0] - filas_vacias]
})
display(tabla_vacias)

,métrica,valor
0,Filas completamente vacías,23
1,Filas con contenido,11867


## 2. Diagnóstico del estado inicial de los datos

### 2.a Número de registros y variables

In [58]:
n_filas, n_cols = datos.shape
tabla_registros = pd.DataFrame({
    "métrica": [
        "Número de registros (crudo, incluye filas vacías)",
        "Número de registros con contenido",
        "Número de variables (sin contar __archivo_origen)"
    ],
    "valor": [n_filas, n_filas - filas_vacias, n_cols - 1]
})
display(tabla_registros)
display(pd.DataFrame({"variable": [c for c in datos.columns if c != '__archivo_origen']}))

,métrica,valor
0,"Número de registros (crudo, incluye filas vacías)",11890
1,Número de registros con contenido,11867
2,Número de variables (sin contar __archivo_origen),17


,variable
0,CODIGO
1,DISTRITO
2,DEPARTAMENTO
3,MUNICIPIO
4,ESTABLECIMIENTO
5,DIRECCION
6,TELEFONO
7,SUPERVISOR
8,DIRECTOR
9,NIVEL


### 2.b Tipo de dato de cada variable

In [59]:
tabla_tipos = pd.DataFrame({
    "variable": datos.columns,
    "tipo_en_pandas": datos.dtypes.astype(str).values
})
display(tabla_tipos)

,variable,tipo_en_pandas
0,CODIGO,str
1,DISTRITO,str
2,DEPARTAMENTO,str
3,MUNICIPIO,str
4,ESTABLECIMIENTO,str
5,DIRECCION,str
6,TELEFONO,str
7,SUPERVISOR,str
8,DIRECTOR,str
9,NIVEL,str


**Nota:** pandas reporta todas las columnas como `object` (texto) porque se cargaron con
`dtype=str` a propósito, para no perder ceros a la izquierda en códigos y evitar que pandas
infiera tipos de forma incorrecta antes de limpiar. Sin embargo, esto NO significa que el tipo
lógico de cada variable sea texto: `CODIGO`, `DISTRITO`, `TELEFONO` deberían tratarse como
identificadores/códigos (texto controlado), mientras que campos como `NIVEL`, `SECTOR`, `AREA`,
`STATUS`, `MODALIDAD`, `JORNADA`, `PLAN`, `DEPARTAMENTO`, `MUNICIPIO`, `DEPARTAMENTAL` son
variables categóricas y deberían convertirse a tipo `category`.

### 2.c Cantidad y porcentaje de valores faltantes por variable

In [60]:
faltantes = datos.drop(columns="__archivo_origen").isna().sum()
faltantes_pct = (faltantes / n_filas * 100).round(2)
tabla_faltantes = pd.DataFrame({"faltantes": faltantes, "porcentaje": faltantes_pct})
tabla_faltantes = tabla_faltantes.sort_values("faltantes", ascending=False).reset_index().rename(columns={"index": "variable"})
display(tabla_faltantes)

,variable,faltantes,porcentaje
0,DIRECTOR,1755,14.76
1,TELEFONO,969,8.15
2,SUPERVISOR,558,4.69
3,DISTRITO,555,4.67
4,DIRECCION,99,0.83
5,ESTABLECIMIENTO,28,0.24
6,CODIGO,23,0.19
7,MUNICIPIO,23,0.19
8,DEPARTAMENTO,23,0.19
9,NIVEL,23,0.19


### 2.d Cantidad de valores únicos por variable

In [61]:
unicos = datos.drop(columns="__archivo_origen").nunique(dropna=True)
tabla_unicos = unicos.sort_values(ascending=False).reset_index()
tabla_unicos.columns = ["variable", "valores_unicos"]
display(tabla_unicos)

,variable,valores_unicos
0,CODIGO,11867
1,DIRECCION,7526
2,ESTABLECIMIENTO,6667
3,TELEFONO,6571
4,DIRECTOR,5561
5,DISTRITO,1681
6,SUPERVISOR,1285
7,MUNICIPIO,352
8,DEPARTAMENTAL,26
9,DEPARTAMENTO,23


### 2.e Registros duplicados exactos

In [62]:
cols_sin_origen = [c for c in datos.columns if c != "__archivo_origen"]
dup_exactos = datos.duplicated(subset=cols_sin_origen).sum()
dup_codigo = datos["CODIGO"].duplicated().sum()
tabla_duplicados = pd.DataFrame({
    "métrica": ["Registros duplicados exactos (ignorando __archivo_origen)", "Códigos (CODIGO) duplicados"],
    "valor": [dup_exactos, dup_codigo]
})
display(tabla_duplicados)

,métrica,valor
0,Registros duplicados exactos (ignorando __arch...,22
1,Códigos (CODIGO) duplicados,22


### 2.f Variables con valores fuera de dominio o inconsistentes

In [63]:
tabla_departamento = datos["DEPARTAMENTO"].value_counts(dropna=False).reset_index()
tabla_departamento.columns = ["DEPARTAMENTO", "conteo"]
display(tabla_departamento)

,DEPARTAMENTO,conteo
0,CIUDAD CAPITAL,2161
1,GUATEMALA,1908
2,SAN MARCOS,724
3,ESCUINTLA,708
4,HUEHUETENANGO,591
5,QUETZALTENANGO,551
6,PETEN,516
7,ALTA VERAPAZ,475
8,SUCHITEPEQUEZ,437
9,CHIMALTENANGO,435


In [64]:
tabla_municipio_capital = datos.loc[datos["DEPARTAMENTO"] == "CIUDAD CAPITAL", "MUNICIPIO"].value_counts().reset_index()
tabla_municipio_capital.columns = ["MUNICIPIO", "conteo"]
display(tabla_municipio_capital)

,MUNICIPIO,conteo
0,ZONA 1,868
1,ZONA 7,236
2,ZONA 12,149
3,ZONA 18,134
4,ZONA 2,118
5,ZONA 6,94
6,ZONA 11,80
7,ZONA 19,65
8,ZONA 3,61
9,ZONA 13,61


In [65]:
for col in ["SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]:
    tabla = datos[col].value_counts(dropna=False).reset_index()
    tabla.columns = [col, "conteo"]
    display(tabla)

,SECTOR,conteo
0,PRIVADO,9891
1,OFICIAL,1499
2,COOPERATIVA,298
3,MUNICIPAL,179
4,NaN,23


,AREA,conteo
0,URBANA,9461
1,RURAL,2403
2,NaN,23
3,SIN ESPECIFICAR,3


,STATUS,conteo
0,ABIERTA,6860
1,CERRADA TEMPORALMENTE,3006
2,CERRADA DEFINITIVAMENTE,1841
3,TEMPORAL TITULOS,110
4,TEMPORAL NOMBRAMIENTO,50
5,NaN,23


,MODALIDAD,conteo
0,MONOLINGUE,11394
1,BILINGUE,473
2,NaN,23


,JORNADA,conteo
0,DOBLE,3772
1,VESPERTINA,3407
2,MATUTINA,3045
3,SIN JORNADA,1099
4,NOCTURNA,415
5,INTERMEDIA,129
6,NaN,23


,PLAN,conteo
0,DIARIO(REGULAR),7452
1,FIN DE SEMANA,2898
2,SEMIPRESENCIAL (FIN DE SEMANA),542
3,SEMIPRESENCIAL (UN DÍA A LA SEMANA),437
4,A DISTANCIA,188
5,SEMIPRESENCIAL,118
6,VIRTUAL A DISTANCIA,70
7,SEMIPRESENCIAL (DOS DÍAS A LA SEMANA),67
8,SABATINO,59
9,DOMINICAL,27


In [66]:
tel = datos["TELEFONO"].dropna()
solo_digitos = tel.apply(lambda x: re.sub(r"[^0-9]", "", x))
tabla_longitudes_tel = solo_digitos.apply(len).value_counts().sort_index().reset_index()
tabla_longitudes_tel.columns = ["longitud_digitos", "conteo"]
display(tabla_longitudes_tel)
ejemplos_tel = pd.DataFrame({"ejemplos_formatos_no_estandar": tel[tel.str.contains(r"[^0-9\-\s]", regex=True, na=False)].unique()[:15]})
display(ejemplos_tel)

,longitud_digitos,conteo
0,2,1
1,4,1
2,5,4
3,6,8
4,7,35
5,8,10673
6,9,3
7,10,12
8,12,4
9,13,1


,ejemplos_formatos_no_estandar
0,77238200/58540260
1,"25763,26725 Y 21568"
2,"25763, 26725 Y 21568"
3,"2325732, 2320075, 2307014"
4,"26725,25763 Y 21568"
5,"251-7968,232-5955"
6,"238-4425,251-4792"
7,238-4283/2384284
8,"2532566,2329566"
9,232-1011 y 251-6422


In [67]:
resumen_placeholder = []
for col in ["DIRECTOR", "DIRECCION", "SUPERVISOR"]:
    vals = datos[col].dropna().unique()
    placeholder = [v for v in vals if set(v.strip()) <= set("-.")]
    resumen_placeholder.append({
        "variable": col,
        "valores_placeholder": len(placeholder),
        "ejemplos": placeholder[:5]
    })
display(pd.DataFrame(resumen_placeholder))

,variable,valores_placeholder,ejemplos
0,DIRECTOR,27,"[--, ---, -, ., ----]"
1,DIRECCION,5,"[--, ---, -----------, ------, .]"
2,SUPERVISOR,0,[]


### 2.g Variables con formatos inconsistentes

In [68]:
cod = datos["CODIGO"].dropna()
patron_ok = cod.str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$").sum()
tabla_codigo = pd.DataFrame({
    "métrica": ["Códigos que siguen el patrón NN-NN-NNNN-NN", "Total de códigos evaluados"],
    "valor": [patron_ok, len(cod)]
})
display(tabla_codigo)

,métrica,valor
0,Códigos que siguen el patrón NN-NN-NNNN-NN,11867
1,Total de códigos evaluados,11867


In [69]:
dist = datos["DISTRITO"].dropna()
tabla_distrito = dist.apply(lambda x: len(x.split("-"))).value_counts().reset_index()
tabla_distrito.columns = ["segmentos", "conteo"]
display(tabla_distrito)
display(pd.DataFrame({"ejemplos_distrito": dist.unique()[:10]}))

,segmentos,conteo
0,3,6226
1,2,5109


,ejemplos_distrito
0,16-006
1,16-01-0926
2,16-01-0927
3,16-01-0924
4,16-01-0925
5,16-01-0923
6,16-031
7,16-01-0937
8,16-026
9,16-01-0938


In [70]:
est = datos["ESTABLECIMIENTO"].dropna()
tabla_establecimiento = pd.DataFrame({
    "métrica": ["Registros con espacios dobles", "Registros con comillas internas"],
    "valor": [est.str.contains("  ").sum(), est.str.contains("\"").sum() + est.str.contains("'").sum()]
})
display(tabla_establecimiento)
display(pd.DataFrame({"ejemplos_con_comillas": est[est.str.contains("\"") | est.str.contains("'")].unique()[:5].tolist()}))

,métrica,valor
0,Registros con espacios dobles,1395
1,Registros con comillas internas,2974


,ejemplos_con_comillas
0,"COLEGIO ""LA INMACULADA"""
1,INSTITUTO NORMAL MIXTO DEL NORTE 'EMILIO ROSAL...
2,"LICEO ""MODERNO LATINO"""
3,"COLEGIO DE INFORMATICA ""CENINFAV"""
4,"COLEGIO EVANGELICO ""LA PATRIA"""


## 2.h Identificación de problemas potenciales de calidad de datos

| Problema potencial | Evidencia |
|---|---|
| Filas completamente vacías al final de cada archivo | Se contabilizaron con código y se muestran en tabla. |
| `DEPARTAMENTO = CIUDAD CAPITAL` no es un departamento real | Se identificó como categoría especial exportada por el portal. |
| `MUNICIPIO` en `CIUDAD CAPITAL` contiene zonas | Se revisó su distribución con una tabla de conteos. |
| `TELEFONO` con múltiples números y separadores inconsistentes | Se revisaron longitudes de dígitos y ejemplos no estándar. |
| `DIRECTOR`, `DIRECCION` y `SUPERVISOR` usan placeholders | Se cuantificaron valores formados por guiones o puntos. |
| `DISTRITO` con formatos mixtos | Se contaron los segmentos separados por guion. |
| `ESTABLECIMIENTO` con espacios dobles y comillas internas | Se midieron ambos casos con conteos y ejemplos. |
| Posibles duplicados parciales por similitud de nombres | Queda planteado como siguiente paso de limpieza. |

Esta evidencia se complementa con las tablas generadas en las celdas anteriores.